In [14]:
import os
print(os.listdir("/kaggle/input/"))

['datasets']


In [15]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets"):
    level = root.replace("/kaggle/input/datasets", "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")

datasets/
  dl32958/
    sroie2019-ocr-rawtext/
      sroie_ocr/
        test/
          box/
          entities/
          ocr/
          img/
        train/
          box/
          entities/
          ocr/
          img/


In [2]:
"""
CELL 1 — Install Unsloth
Run this first, then RESTART RUNTIME
"""
!pip install -q unsloth
!pip install -q datasets

# ═══════════════════════════════════════════════════════════════
# CELL 2 — Verify Dataset Paths
# ═══════════════════════════════════════════════════════════════
import os
print("Train OCR files:", len(os.listdir("/kaggle/input/datasets/dl32958/sroie2019-ocr-rawtext/sroie_ocr/train/ocr")))
print("Test  OCR files:", len(os.listdir("/kaggle/input/datasets/dl32958/sroie2019-ocr-rawtext/sroie_ocr/test/ocr")))
print("✓ Dataset paths verified!")

# ═══════════════════════════════════════════════════════════════
# CELL 3 — Imports
# ═══════════════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

import torch
import json
import re

from huggingface_hub import login
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from transformers import TrainerCallback

print("✓ All imports successful!")
print(f"✓ Torch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

# ═══════════════════════════════════════════════════════════════
# CELL 4 — Login
# ═══════════════════════════════════════════════════════════════
print("Logging in to Hugging Face...")
login(token="[TOKEN]")

print("✓ Logged in!\n")

# ═══════════════════════════════════════════════════════════════
# CELL 5 — Config
# ═══════════════════════════════════════════════════════════════
MODEL_ID    = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_DIR  = "./llama32-invoice-sft"
MAX_SEQ_LEN = 512

TRAIN_OCR_DIR  = "/kaggle/input/datasets/dl32958/sroie2019-ocr-rawtext/sroie_ocr/train/ocr"
TEST_OCR_DIR   = "/kaggle/input/datasets/dl32958/sroie2019-ocr-rawtext/sroie_ocr/test/ocr"
TRAIN_GT_DIR   = "/kaggle/input/datasets/dl32958/sroie2019-ocr-rawtext/sroie_ocr/train/entities"
TEST_RESULT_DIR = "./test_results"
os.makedirs(TEST_RESULT_DIR, exist_ok=True)

SYSTEM_PROMPT = (
    "You are an invoice extraction assistant. "
    "Given raw OCR text from an invoice, extract the following fields "
    "and return ONLY a valid JSON object: company, date, address, total."
)

# ═══════════════════════════════════════════════════════════════
# CELL 6 — Format Prompt
# ═══════════════════════════════════════════════════════════════
def format_prompt(ocr_text, gt_json=None):
    p = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}<|eot_id|>\n"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{ocr_text.strip()}<|eot_id|>\n"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    if gt_json:
        p += f"{gt_json}<|eot_id|>"
    return p

# ═══════════════════════════════════════════════════════════════
# CELL 7 — Load Train Dataset from SROIE OCR txt files
# ═══════════════════════════════════════════════════════════════
print("Loading training data from SROIE OCR files...")
train_samples = []

# Debug: print first ground truth file to see its format
sample_gt_files = sorted(os.listdir(TRAIN_GT_DIR))[:1]
for sf in sample_gt_files:
    with open(os.path.join(TRAIN_GT_DIR, sf), "r", encoding="utf-8", errors="ignore") as f:
        print(f"Sample GT file ({sf}):\n{f.read()}\n")

for fname in sorted(os.listdir(TRAIN_OCR_DIR)):
    if not fname.endswith(".txt"):
        continue

    ocr_path = os.path.join(TRAIN_OCR_DIR, fname)
    gt_path  = os.path.join(TRAIN_GT_DIR, fname)

    if not os.path.exists(gt_path):
        continue

    with open(ocr_path, "r", encoding="utf-8", errors="ignore") as f:
        ocr_text = f.read().strip()
    with open(gt_path, "r", encoding="utf-8", errors="ignore") as f:
        gt_raw = f.read().strip()

    gt_dict = {}

    # Try standard JSON first
    try:
        gt_dict = json.loads(gt_raw)
    except json.JSONDecodeError:
        # Try key:value per line format
        for line in gt_raw.splitlines():
            line = line.strip()
            if not line:
                continue
            if ":" in line:
                key, _, val = line.partition(":")
                key = key.strip().lower().replace('"', '').replace("'", '')
                val = val.strip().replace('"', '').replace("'", '')
                if key in ["company", "date", "address", "total"]:
                    gt_dict[key] = val
            # Try key=value format as fallback
            elif "=" in line:
                key, _, val = line.partition("=")
                key = key.strip().lower()
                val = val.strip()
                if key in ["company", "date", "address", "total"]:
                    gt_dict[key] = val

    if not gt_dict:
        print(f"  Skipping {fname} — could not parse ground truth")
        continue

    gt_str = json.dumps(gt_dict, indent=2)
    train_samples.append({"text": format_prompt(ocr_text, gt_str)})

print(f"Total samples loaded: {len(train_samples)}")
if len(train_samples) > 0:
    print(f"Sample entry:\n{train_samples[0]['text'][:300]}")

# 90/10 split for train/val
split       = int(len(train_samples) * 0.9)
train_dataset = Dataset.from_list(train_samples[:split])
val_dataset   = Dataset.from_list(train_samples[split:])

print(f"✓ Train samples : {len(train_dataset)}")
print(f"✓ Val   samples : {len(val_dataset)}")

# ═══════════════════════════════════════════════════════════════
# CELL 8 — Load Model + Tokenizer via Unsloth
# ═══════════════════════════════════════════════════════════════
print("\nLoading model and tokenizer via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.float16,
    load_in_4bit=True,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✓ Model and tokenizer loaded!")

# ═══════════════════════════════════════════════════════════════
# CELL 9 — Attach LoRA
# ═══════════════════════════════════════════════════════════════
print("Attaching LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing=True,
)
print("✓ LoRA attached!")

# ═══════════════════════════════════════════════════════════════
# CELL 10 — Progress Callback
# ═══════════════════════════════════════════════════════════════
class ProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            print(f"Step {state.global_step}/{state.max_steps} - Loss: {logs['loss']:.4f}")
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n✓ Epoch {int(state.epoch)} completed!\n")

# ═══════════════════════════════════════════════════════════════
# CELL 11 — SFT Fine-Tuning
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("STARTING SFT FINE-TUNING...")
print("=" * 60)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
    callbacks=[ProgressCallback()],
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✓ Fine-tuned model saved to {OUTPUT_DIR}")

# ═══════════════════════════════════════════════════════════════
# CELL 12 — Test on all test OCR files, save results as .txt
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TESTING ON SROIE TEST SET...")
print("=" * 60)

FastLanguageModel.for_inference(model)
model.eval()

test_files = sorted([f for f in os.listdir(TEST_OCR_DIR) if f.endswith(".txt")])
print(f"Total test files: {len(test_files)}\n")

total   = len(test_files)
success = 0

for i, fname in enumerate(test_files):
    ocr_path = os.path.join(TEST_OCR_DIR, fname)

    with open(ocr_path, "r", encoding="utf-8", errors="ignore") as f:
        ocr_text = f.read().strip()

    # Build prompt without ground truth
    test_prompt = format_prompt(ocr_text)
    inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded   = tokenizer.decode(output[0], skip_special_tokens=True)
    raw_reply = decoded.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

    # Parse predicted JSON — handle all edge cases
    predicted = {}
    try:
        predicted = json.loads(raw_reply)
    except (json.JSONDecodeError, ValueError):
        try:
            # Fix missing closing brace
            fixed = raw_reply.strip()
            if not fixed.endswith("}"):
                fixed += "}"
            predicted = json.loads(fixed)
        except (json.JSONDecodeError, ValueError):
            try:
                # Extract JSON block using regex
                match = re.search(r'\{[^{}]*\}', fixed, re.DOTALL)
                if match:
                    predicted = json.loads(match.group())
            except (json.JSONDecodeError, ValueError, AttributeError):
                # Last resort: extract key-value pairs manually
                for field in ["company", "date", "address", "total"]:
                    pattern = rf'"{field}"\s*:\s*"([^"]*)"'
                    m = re.search(pattern, raw_reply, re.IGNORECASE)
                    if m:
                        predicted[field] = m.group(1)
    if not predicted:
        print(f"  Warning: could not parse output for {fname}, saving empty result")

    # Save result as .txt file with same filename
    result_path = os.path.join(TEST_RESULT_DIR, fname)
    with open(result_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(predicted, indent=4))

    if predicted:
        success += 1

    print(f"[{i+1}/{total}] {fname} → saved to {result_path}")
    print(f"  Predicted: {predicted}\n")

print("=" * 60)
print(f"✓ Testing complete! {success}/{total} files processed successfully")
print(f"✓ All results saved in: {TEST_RESULT_DIR}")
print("=" * 60)

Train OCR files: 626
Test  OCR files: 347
✓ Dataset paths verified!
✓ All imports successful!
✓ Torch version: 2.10.0+cu128
✓ CUDA available: True
Logging in to Hugging Face...
✓ Logged in!

Loading training data from SROIE OCR files...
Sample GT file (X00016469612.txt):
{
    "company": "BOOK TA .K (TAMAN DAYA) SDN BHD",
    "date": "25/12/2018",
    "address": "NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",
    "total": "9.00"
}

Total samples loaded: 626
Sample entry:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an invoice extraction assistant. Given raw OCR text from an invoice, extract the following fields and return ONLY a valid JSON object: company, date, address, total.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
tan woon yann


✓ Train samples : 563
✓ Val   samples : 63

Loading model and tokenizer via Unsloth...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/563 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/63 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 563 | Num Epochs = 3 | Total steps = 213
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Epoch,Training Loss,Validation Loss
1,2.039400,2.035652
2,1.752600,1.838694
3,1.657700,1.811513


...

[319/347] X51007846355.txt → saved to ./test_results/X51007846355.txt
  Predicted: {'company': 'AEON CO. (M) BHD', 'date': '17/06/2018', 'address': '3RD FLR, AEON TAMAN MALURI SC JLN JEJAKA, TAMAN MALURI CHERAS, 55100 KUALA LUMPUR', 'total': '8.95'}

[320/347] X51007846358.txt → saved to ./test_results/X51007846358.txt
  Predicted: {}

[321/347] X51007846360.txt → saved to ./test_results/X51007846360.txt
  Predicted: {}

[322/347] X51007846361.txt → saved to ./test_results/X51007846361.txt
  Predicted: {'company': '118 MJ MOOKATA HOUSE', 'date': '11-06-2018', 'address': 'NO.7G, JALAN PERMAS 11, BANDAR BARU PERMAS JAYA, 81750 MASAI, JOHOR.', 'total': '102.40'}

[323/347] X51007846371.txt → saved to ./test_results/X51007846371.txt
  Predicted: {'company': 'UNIHAKKA INTERNATIONAL SDN BHD', 'date': '13 JUN 2018', 'address': '12, JALAN TAMPOI 7/4,KAWASAN PERINDUSTRIAN TAMPOI,81200 JOHOR BAHRU,JOHOR', 'total': 'RM8.85'}



Unsloth: Input IDs of shape torch.Size([1, 617]) with length 617 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.


[324/347] X51007846372.txt → saved to ./test_results/X51007846372.txt
  Predicted: {'company': 'UNIHAKKA INTERNATIONAL SDN BHD', 'date': '14 JUN 2018', 'address': '12, JALAN TAMPOI 7/4,KAWASAN PERINDUSTRIAN TAMPOI,81200 JOHOR BAHRU,JOHOR', 'total': 'RM10.15'}

[325/347] X51007846379.txt → saved to ./test_results/X51007846379.txt
  Predicted: {}

[326/347] X51007846387.txt → saved to ./test_results/X51007846387.txt
  Predicted: {'company': 'BOOK TALK (MUTIARA RINI) SDN BHD', 'date': '21/6/2018', 'address': 'NO. 53 & 55, JALAN UTAMA 34, MUTIARA RINI, SKUDALI, JOHOR.', 'total': '9.30'}

[327/347] X51007846392.txt → saved to ./test_results/X51007846392.txt
  Predicted: {'company': 'UNIHAKKA INTERNATIONAL SDN BHD', 'date': '19 JUN 2018', 'address': '12, JALAN TAMPOI 7/4,KAWASAN PERINDUSTRIAN TAMPOI,81200 JOHOR BAHRU,JOHOR', 'total': 'RM8.45'}

[328/347] X51007846396.txt → saved to ./test_results/X51007846396.txt
  Predicted: {'company': 'GERBANG ALAF RESTAURANTS SDN BHD', 'date': '24/06/201

Unsloth: Input IDs of shape torch.Size([1, 600]) with length 600 > the model's max sequence length of 512.
We shall truncate it ourselves. It's imperative if you correct this issue first.


[337/347] X51008042786.txt → saved to ./test_results/X51008042786.txt
  Predicted: {'company': 'GARBANG ALAF RESTAURANTS SDN BHD', 'date': '30/05/2018', 'address': 'LAVE 6, BANGUNAN TH, DAMANSARA UPTOWN', 'total': '19.00'}

[338/347] X51008042787.txt → saved to ./test_results/X51008042787.txt
  Predicted: {}

[339/347] X51008099047.txt → saved to ./test_results/X51008099047.txt
  Predicted: {'company': 'RESTORAN WAN SHENG', 'date': '21-06-2018', 'address': 'NO.2, JALAN TEMENGGUNG 19/9, SEKSYEN 9, BANDAR MAHKOTA CHERAS, 43200 CHERAS, SELANGOR', 'total': '4.60'}

[340/347] X51008099086.txt → saved to ./test_results/X51008099086.txt
  Predicted: {'company': 'RESTORAN WAN SHENG', 'date': '29-06-2018', 'address': 'NO.2, JALAN TEMENGGUNG 19/9, SEKSYEN 9, BANDAR MAHKOTA CHERAS, 43200 CHERAS, SELANGOR', 'total': '7.20'}

[341/347] X51008099088.txt → saved to ./test_results/X51008099088.txt
  Predicted: {'company': 'RESTORAN WAN SHENG', 'date': '27-06-2018', 'address': 'NO.2, JALAN TEMENGGUNG 1

In [3]:
print("\n" + "=" * 60)
print("EVALUATING RESULTS...")
print("=" * 60)

TEST_GT_DIR = "/kaggle/input/datasets/dl32958/sroie2019-ocr-rawtext/sroie_ocr/test/entities"
FIELDS      = ["company", "date", "address", "total"]

# Counters
exact_match   = {f: 0 for f in FIELDS}
total_count   = {f: 0 for f in FIELDS}
files_scored  = 0
files_skipped = 0

def parse_gt(gt_raw):
    """Parse ground truth file — try JSON first, then key:value"""
    gt_dict = {}
    try:
        gt_dict = json.loads(gt_raw)
    except json.JSONDecodeError:
        for line in gt_raw.splitlines():
            line = line.strip()
            if not line:
                continue
            if ":" in line:
                key, _, val = line.partition(":")
                key = key.strip().lower().replace('"','').replace("'",'')
                val = val.strip().replace('"','').replace("'",'')
                if key in FIELDS:
                    gt_dict[key] = val
    return gt_dict

for fname in sorted(os.listdir(TEST_RESULT_DIR)):
    if not fname.endswith(".txt"):
        continue

    pred_path = os.path.join(TEST_RESULT_DIR, fname)
    gt_path   = os.path.join(TEST_GT_DIR, fname)

    if not os.path.exists(gt_path):
        files_skipped += 1
        continue

    with open(pred_path, "r", encoding="utf-8", errors="ignore") as f:
        pred_raw = f.read().strip()
    with open(gt_path, "r", encoding="utf-8", errors="ignore") as f:
        gt_raw = f.read().strip()

    try:
        predicted = json.loads(pred_raw) if pred_raw else {}
    except json.JSONDecodeError:
        predicted = {}

    gt_dict = parse_gt(gt_raw)

    if not gt_dict or not predicted:
        files_skipped += 1
        continue

    files_scored += 1
    for field in FIELDS:
        gt_val   = gt_dict.get(field, "").strip().upper()
        pred_val = predicted.get(field, "").strip().upper()
        if gt_val:
            total_count[field] += 1
            if gt_val == pred_val:
                exact_match[field] += 1

# ── Print Results ─────────────────────────────────────────────
print(f"\nFiles evaluated : {files_scored}")
print(f"Files skipped   : {files_skipped}")
print("\nField-Level Exact Match Accuracy:")
print("-" * 40)

overall_correct = 0
overall_total   = 0
for field in FIELDS:
    correct = exact_match[field]
    tot     = total_count[field]
    acc     = (correct / tot * 100) if tot > 0 else 0
    overall_correct += correct
    overall_total   += tot
    print(f"  {field.upper():<12} : {correct}/{tot} = {acc:.2f}%")

overall_acc = (overall_correct / overall_total * 100) if overall_total > 0 else 0
print("-" * 40)
print(f"  {'OVERALL':<12} : {overall_correct}/{overall_total} = {overall_acc:.2f}%")
print("=" * 60)

# ── Save evaluation summary ───────────────────────────────────
summary = {
    "files_evaluated": files_scored,
    "files_skipped":   files_skipped,
    "field_accuracy": {
        f: f"{exact_match[f]}/{total_count[f]} = {(exact_match[f]/total_count[f]*100):.2f}%"
        if total_count[f] > 0 else "N/A"
        for f in FIELDS
    },
    "overall_accuracy": f"{overall_correct}/{overall_total} = {overall_acc:.2f}%"
}
with open("evaluation_summary.txt", "w") as f:
    f.write(json.dumps(summary, indent=4))
print("\n✓ Evaluation summary saved to evaluation_summary.txt")


EVALUATING RESULTS...

Files evaluated : 329
Files skipped   : 18

Field-Level Exact Match Accuracy:
----------------------------------------
  COMPANY      : 222/329 = 67.48%
  DATE         : 240/329 = 72.95%
  ADDRESS      : 123/329 = 37.39%
  TOTAL        : 223/329 = 67.78%
----------------------------------------
  OVERALL      : 808/1316 = 61.40%

✓ Evaluation summary saved to evaluation_summary.txt


In [7]:
import zipfile
import os

# Zip the test_results folder
with zipfile.ZipFile('/kaggle/working/test_results.zip', 'w') as zipf:
    for fname in os.listdir('/kaggle/working/test_results'):
        zipf.write(os.path.join('/kaggle/working/test_results', fname), fname)

print("✓ Zipped successfully!")

# Download link
from IPython.display import FileLink
FileLink('/kaggle/working/test_results.zip')

✓ Zipped successfully!


/kaggle/working/test_results.zip

In [9]:
from IPython.display import Javascript
Javascript("window.open('/kaggle/working/test_results.zip')")

<IPython.core.display.Javascript object>

In [10]:
import os
import json

# Check if file exists
print(os.path.exists('/kaggle/working/evaluation_summary.txt'))
print(os.listdir('/kaggle/working'))

True
['unsloth_compiled_cache', 'test_results.zip', '.virtual_documents', 'evaluation_summary.txt', 'huggingface_tokenizers_cache', 'llama32-invoice-sft', 'test_results']
